# Hierarchical Process

CrewAI learning notebook — companion to the roadmap. Platform: [Agent Foundry](https://agent-foundry-pi.vercel.app)

In [ ]:
!pip install -q crewai crewai-tools

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

## Hierarchical crew with `manager_llm`

CrewAI **auto-creates a manager** powered by the LLM you pass. Tasks describe **what** to deliver; the manager **delegates** to specialists (`allow_delegation=False` on workers) based on roles and goals—not a fixed per-task agent binding.

In [ ]:
from crewai import Agent, Task, Crew, Process

researcher = Agent(
    role="Researcher",
    goal="Gather accurate, sourced facts and concise notes on the topic.",
    backstory="You are rigorous about facts and avoid speculation.",
    verbose=True,
    allow_delegation=False,
)

analyst = Agent(
    role="Analyst",
    goal="Turn raw information into clear insights, risks, and recommendations.",
    backstory="You structure messy inputs and call out assumptions.",
    verbose=True,
    allow_delegation=False,
)

writer = Agent(
    role="Writer",
    goal="Produce polished prose that matches the requested format.",
    backstory="You edit for clarity and audience-appropriate tone.",
    verbose=True,
    allow_delegation=False,
)

task_a = Task(
    description="Understand the landscape for {topic}: key players, trends, and one open question.",
    expected_output="Short structured notes suitable for further analysis.",
)

task_b = Task(
    description="From the team's findings so far, identify top risks and opportunities for {topic}.",
    expected_output="Bulleted risks and opportunities with one-line rationale each.",
)

task_c = Task(
    description="Produce a brief executive summary for stakeholders on {topic}.",
    expected_output="Two or three tight paragraphs, no fluff.",
)

crew_auto_manager = Crew(
    agents=[researcher, analyst, writer],
    tasks=[task_a, task_b, task_c],
    process=Process.hierarchical,
    manager_llm="gpt-4o",
    verbose=True,
)

result_auto = crew_auto_manager.kickoff(inputs={"topic": "CrewAI hierarchical crews"})
print(result_auto.raw)

## Custom `manager_agent`

Same workers and tasks, but the **lead** is an explicit `Agent` with its own **role**, **goal**, and **backstory**—useful when you want a specific coordination style (e.g. consulting lead, strict editor).

In [ ]:
project_manager = Agent(
    role="Engagement Lead",
    goal="Deliver a coherent, client-ready brief by delegating to research, analysis, and writing experts—and reject work that misses the brief.",
    backstory=(
        "You have run consulting engagements for a decade. "
        "You insist on clarity, traceable reasoning, and one voice in the final summary."
    ),
    verbose=True,
    allow_delegation=True,
)

task_a2 = Task(
    description="Understand the landscape for {topic}: key players, trends, and one open question.",
    expected_output="Short structured notes suitable for further analysis.",
)

task_b2 = Task(
    description="From the team's findings so far, identify top risks and opportunities for {topic}.",
    expected_output="Bulleted risks and opportunities with one-line rationale each.",
)

task_c2 = Task(
    description="Produce a brief executive summary for stakeholders on {topic}.",
    expected_output="Two or three tight paragraphs, no fluff.",
)

crew_custom_manager = Crew(
    agents=[researcher, analyst, writer],
    tasks=[task_a2, task_b2, task_c2],
    process=Process.hierarchical,
    manager_agent=project_manager,
    verbose=True,
)

result_custom = crew_custom_manager.kickoff(inputs={"topic": "CrewAI hierarchical crews"})
print(result_custom.raw)

## Key takeaways

- A **hierarchical** crew uses a **manager** to **delegate**, **validate**, and **resolve** unclear handoffs; workers usually set **`allow_delegation=False`**.
- **Assignment is dynamic**: tasks define outcomes; the manager maps work to specialists by **capability**, not a single frozen task→agent table.
- Use **`manager_llm="gpt-4o"`** for a built-in manager, or **`manager_agent=...`** for a **custom** lead with explicit role/goal.
- Prefer **hierarchical** for **complex**, **context-dependent** work; prefer **sequential** for **fixed-order** pipelines.